# MySQL database checks

This notebook runs read-only validation queries against the local `mancino_analytics` database. Run the cells after the schema has been created and the CSV data has been loaded. The notebook stores each cell's output when you save it after execution.

The expected row counts below are based on the current source CSV files. The MySQL password is requested at runtime and is not written into the notebook.

## 1. Connect to local MySQL

The notebook uses `mysql-connector-python` and pandas. If the connection fails because `mancino_analytics` does not exist yet, first run the schema and data loader.

In [1]:
from getpass import getpass
import mysql.connector
import pandas as pd
from IPython.display import display

mysql_password = getpass('MySQL root password: ')
connection = mysql.connector.connect(
    host='localhost',
    port=3306,
    user='root',
    password=mysql_password,
    database='mancino_analytics',
)
del mysql_password
print('Connected to MySQL database:', connection.database)

Connected to MySQL database: mancino_analytics


## 2. Query helper

This helper runs a `SELECT` statement and displays its result as a table. The cells below contain the SQL being checked.

In [2]:
def show_query(sql):
    cursor = connection.cursor()
    try:
        cursor.execute(sql)
        result = pd.DataFrame(cursor.fetchall(), columns=[column[0] for column in cursor.description])
        display(result)
        return result
    finally:
        cursor.close()

## 3. Loaded row counts

Expected counts: 5 stores, 1,500 products, 167,823 transactions, and 504,000 line items.

In [3]:
row_counts = show_query('''
SELECT 'stores' AS table_name, COUNT(*) AS actual_count FROM stores
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
UNION ALL SELECT 'transaction_line_items', COUNT(*) FROM transaction_line_items
''')
expected_counts = {
    'stores': 5,
    'products': 1500,
    'transactions': 167823,
    'transaction_line_items': 504000,
}
row_counts['expected_count'] = row_counts['table_name'].map(expected_counts)
row_counts['matches_source'] = row_counts['actual_count'] == row_counts['expected_count']
display(row_counts)

,table_name,actual_count
0,stores,5
1,products,1500
2,transactions,167823
3,transaction_line_items,504000


,table_name,actual_count,expected_count,matches_source
0,stores,5,5,True
1,products,1500,1500,True
2,transactions,167823,167823,True
3,transaction_line_items,504000,504000,True


## 4. Foreign-key relationship checks

Each issue count should be zero. The schema's foreign keys should also prevent orphan records from being inserted.

In [4]:
show_query('''
SELECT 'transactions_without_store' AS check_name, COUNT(*) AS issue_count
FROM transactions t LEFT JOIN stores s ON s.store_id = t.store_id
WHERE s.store_id IS NULL
UNION ALL
SELECT 'line_items_without_transaction', COUNT(*)
FROM transaction_line_items li LEFT JOIN transactions t ON t.transaction_id = li.transaction_id
WHERE t.transaction_id IS NULL
UNION ALL
SELECT 'line_items_without_product', COUNT(*)
FROM transaction_line_items li LEFT JOIN products p ON p.product_id = li.product_id
WHERE p.product_id IS NULL
''')

,check_name,issue_count
0,transactions_without_store,0
1,line_items_without_transaction,0
2,line_items_without_product,0


,check_name,issue_count
0,transactions_without_store,0
1,line_items_without_transaction,0
2,line_items_without_product,0


## 5. Line-item arithmetic and cost coverage

Both arithmetic mismatch counts and the product count without a unit cost should be zero. A two-cent tolerance is allowed for monetary rounding.

In [5]:
show_query('''
SELECT
    SUM(ABS(qty * unit_price - line_subtotal) > 0.02) AS subtotal_mismatches,
    SUM(ABS(line_subtotal - line_discount - line_total) > 0.02) AS net_total_mismatches
FROM transaction_line_items
''')
show_query('''
SELECT COUNT(*) AS products_without_unit_cost
FROM products
WHERE unit_cost IS NULL
''')

,subtotal_mismatches,net_total_mismatches
0,0,0


,products_without_unit_cost
0,0


,products_without_unit_cost
0,0


## 6. Transaction header-to-line reconciliation

This is diagnostic rather than an automatic pass/fail check. It compares header subtotal with gross line subtotals, and header total with net line totals plus tax. Differences need interpretation based on the dataset's field definitions.

In [6]:
show_query('''
WITH line_totals AS (
    SELECT transaction_id,
           SUM(line_subtotal) AS gross_line_subtotal,
           SUM(line_total) AS net_line_total
    FROM transaction_line_items
    GROUP BY transaction_id
)
SELECT
    COUNT(*) AS matched_transactions,
    SUM(ABS(t.subtotal - lt.gross_line_subtotal) <= 0.02) AS header_subtotal_matches_gross_lines,
    SUM(ABS(t.total - lt.net_line_total - t.tax) <= 0.02) AS header_total_matches_net_lines_plus_tax,
    SUM(ABS(t.subtotal - lt.gross_line_subtotal) > 0.02) AS header_subtotal_differences,
    SUM(ABS(t.total - lt.net_line_total - t.tax) > 0.02) AS header_total_differences
FROM transactions t
JOIN line_totals lt ON lt.transaction_id = t.transaction_id
''')

,matched_transactions,header_subtotal_matches_gross_lines,header_total_matches_net_lines_plus_tax,header_subtotal_differences,header_total_differences
0,167823,167823,167823,0,0


,matched_transactions,header_subtotal_matches_gross_lines,header_total_matches_net_lines_plus_tax,header_subtotal_differences,header_total_differences
0,167823,167823,167823,0,0


## 7. Monthly revenue and estimated gross profit

This uses discounted `line_total` as line revenue and `qty × unit_cost` as estimated cost of goods sold. The full timestamp remains stored; this query groups by calendar month for analysis.

In [7]:
monthly_results = show_query('''
SELECT
    DATE_FORMAT(t.txn_ts, '%Y-%m') AS month,
    SUM(li.line_total) AS net_revenue,
    SUM(li.line_total - li.qty * p.unit_cost) AS estimated_gross_profit
FROM transaction_line_items li
JOIN transactions t ON t.transaction_id = li.transaction_id
JOIN products p ON p.product_id = li.product_id
GROUP BY DATE_FORMAT(t.txn_ts, '%Y-%m')
ORDER BY month
''')

,month,net_revenue,estimated_gross_profit
0,2025-01,461624.69,144646.36000
1,2025-02,416110.21,129436.78000
2,2025-03,448353.61,137625.99000
3,2025-04,426447.09,133347.12000
4,2025-05,465735.01,145898.74000
5,2025-06,435596.34,135000.39000
6,2025-07,448410.62,137733.15000
7,2025-08,465043.65,143545.25000
8,2025-09,448601.60,139061.22000
9,2025-10,444993.55,136250.87000


## 8. Close the connection

Run this after finishing the checks. To rerun queries later, execute the connection and helper cells again first.

In [8]:
connection.close()